# 07 · Hyperparameter tuning (Optuna)

Protocol (no test leakage):
1. **Expanding-window CV inside the training years:** fit on years < Y, validate on Y, for Y = 2020, 2021, 2022, 2023. The objective is the mean validation RMSE.
2. Tree-structured Parzen search with median pruning (bad trials stop after 1-2 folds).
3. Refit the best params on 2016 to 2023 and score **once** on 2024 to 2025.

The search space includes **Huber loss** (with a threshold on the right scale) because the target is heavy-tailed (notebook 02).

This notebook runs a short study so it finishes on a laptop CPU. For the real search, use the script (resumable, GPU-capable):
```
python scripts/tune_gbm.py --trials 300 --storage sqlite:///results/tuning.db                 # LightGBM, CPU
python scripts/tune_gbm.py --kind xgb --device cuda --trials 500 --storage sqlite:///results/tuning.db   # XGBoost, GPU
python scripts/tune_gbm.py --target nlri --depth 10 --trials 200                               # nitrate index
```

In [ ]:
# Setup: works locally (pip install -e . from the repo root) and on Google Colab
import sys, warnings
warnings.filterwarnings("ignore")
if "google.colab" in sys.modules:
    # Upload soilcast.zip to Google Drive (MyDrive) first
    from google.colab import drive
    drive.mount("/content/drive")
    !unzip -q -o /content/drive/MyDrive/soilcast.zip -d /content
    %cd /content/soilcast
    !pip install -q -e ".[gpu-trees]"
    sys.path.insert(0, "/content/soilcast/src")

import numpy as np, pandas as pd, matplotlib.pyplot as plt
from soilcast.config import PROCESSED, RAW, RESULTS
from soilcast.plotting import set_style, model_color, SERIES, MUTED
set_style()
pd.set_option("display.width", 160, "display.max_columns", 30, "display.precision", 3)
RESULTS.mkdir(exist_ok=True)

In [ ]:
import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)
from soilcast.features import FEATURE_SETS, sm_valid
from soilcast.tuning import run_study, params_from_trial
from soilcast.evaluate import time_split, sm_metrics
from soilcast.models import make_model
N_TRIALS = 15          # raise to 200+ on Colab / RTX
MAX_ROWS = 60_000      # subsample per fold for speed; set None for full data
KIND, DEVICE = "gbm", "cpu"   # KIND="xgb", DEVICE="cuda" on a GPU
d = pd.read_parquet(PROCESSED / "features_5cm_perfect.parquet"); v = d[sm_valid(d)]
cols = FEATURE_SETS["v2_engineered"]

In [ ]:
study = run_study(v, cols, "sm_t24", "sm", kind=KIND, n_trials=N_TRIALS, device=DEVICE, max_rows=MAX_ROWS)
print("best CV RMSE (pp):", round(100 * study.best_value, 4))
best = params_from_trial(study.best_params, KIND); best

In [ ]:
from optuna.visualization.matplotlib import plot_optimization_history, plot_param_importances
plot_optimization_history(study); plt.title("CV RMSE by trial")
if len([t for t in study.trials if t.state.name == "COMPLETE"]) > 3:
    plot_param_importances(study)
study.trials_dataframe()[["number", "value", "state", "params_objective", "params_num_leaves", "params_learning_rate", "params_n_estimators"]].sort_values("value").head(10)

## Refit on all training years, score once on the test years

In [ ]:
tr, te = time_split(v)
default = make_model(KIND, device=DEVICE).fit(tr[cols], tr.sm_t24 - tr.sm)
tuned = make_model(KIND, best, DEVICE).fit(tr[cols], tr.sm_t24 - tr.sm)
res = sm_metrics(te.sm_t24, {"persistence": te.sm, "gbm default": te.sm + default.predict(te[cols]),
                             "tuned " + KIND: te.sm + tuned.predict(te[cols])}, te.sm, te.fc_precip_24h)
res.to_csv(RESULTS / "07_tuned_vs_default_5cm.csv", index=False); res

## What else to tune (ideas, in rough order of expected payoff)
- **Loss:** Huber threshold, or a two-part model (P(rain response) × size), since rain windows dominate error
- **Feature windows:** past window length (24 h vs 72 h), rain thresholds, 30-day floor window
- **Per-texture models** or texture-stratified sample weights
- **Training with forecast noise** (notebook 04) at different noise levels, tuned on NBM-period validation
- **Multi-depth model:** one model with depth as a feature, trained on 5/10/20 cm jointly